In [12]:
import os
import requests
from dotenv import load_dotenv

# Load private settings from the .env file (never uploaded to GitHub)
load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}
print("User-Agent loaded:", HEADERS["User-Agent"] is not None)

CIK = "0001436126"  # Mistras Group (10 digits, with leading zeros)
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"

response = requests.get(url, headers=HEADERS, timeout=30)
print("Status code:", response.status_code)

data = response.json()
print("Company:", data["entityName"])

User-Agent loaded: True
Status code: 200
Company: Mistras Group, Inc.


In [13]:
revenue_facts = data["facts"]["us-gaap"]["RevenueFromContractWithCustomerExcludingAssessedTax"]["units"]["USD"]

print("Number of revenue entries:", len(revenue_facts))
print("First entry:", revenue_facts[0])

Number of revenue entries: 187
First entry: {'start': '2015-06-01', 'end': '2015-12-31', 'val': 427913000, 'accn': '0001436126-19-000014', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2019-03-18'}


In [14]:
import pandas as pd

# Turn the list of entries into a table
rev = pd.DataFrame(revenue_facts)

# Convert date text into real dates so we can do date math
rev["start"] = pd.to_datetime(rev["start"])
rev["end"] = pd.to_datetime(rev["end"])
rev["filed"] = pd.to_datetime(rev["filed"])

# Length of each period in days
rev["days"] = (rev["end"] - rev["start"]).dt.days

# Rules 1 and 2: annual reports only, and only full-year periods
annual = rev[(rev["form"] == "10-K") & (rev["days"].between(350, 380))]

# Rule 3: for each period end date, keep the most recently filed version
annual = annual.sort_values("filed").drop_duplicates(subset="end", keep="last")

# Rule 4: convert to thousands, and keep the useful columns
annual["revenue_000s"] = annual["val"] / 1000
annual = annual[["end", "revenue_000s", "accn", "filed", "days"]].sort_values("end")

annual

,end,revenue_000s,accn,filed,days
1,2016-05-31,719181.0,0001436126-19-000014,2019-03-18,365
2,2016-12-31,684762.0,0001436126-19-000014,2019-03-18,365
12,2017-12-31,700970.0,0001436126-20-000018,2020-03-27,364
32,2018-12-31,742354.0,0001436126-21-000013,2021-03-16,364
56,2019-12-31,748586.0,0001436126-22-000013,2022-03-14,364
84,2020-12-31,592571.0,0001436126-23-000014,2023-03-15,365
115,2021-12-31,677131.0,0001436126-24-000025,2024-03-11,364
142,2022-12-31,687373.0,0001436126-25-000026,2025-03-11,364
161,2023-12-31,705473.0,0001628280-26-016765,2026-03-11,364
174,2024-12-31,729640.0,0001628280-26-016765,2026-03-11,365


In [15]:
# Revenue from the Excel answer key ($000s), typed from the 10-Ks by hand
answer_key = {2025: 724024, 2024: 729640, 2023: 705473, 2022: 687373}

for year, expected in answer_key.items():
    actual = annual.loc[annual["end"].dt.year == year, "revenue_000s"].iloc[0]
    result = "MATCH" if actual == expected else "MISMATCH"
    print(year, "| answer key:", expected, "| API:", actual, "|", result)

2025 | answer key: 724024 | API: 724024.0 | MATCH
2024 | answer key: 729640 | API: 729640.0 | MATCH
2023 | answer key: 705473 | API: 705473.0 | MATCH
2022 | answer key: 687373 | API: 687373.0 | MATCH


In [16]:
def get_annual_values(data, candidate_tags, fye_month):
    """Return one value per fiscal year (in thousands), combining candidate tags in priority order."""
    facts = data["facts"]["us-gaap"]
    tables = []

    for priority, tag in enumerate(candidate_tags):
        if tag not in facts:
            continue
        units = facts[tag]["units"]
        unit_name = list(units.keys())[0]        # "USD" for money, "shares" for share counts
        df = pd.DataFrame(units[unit_name])

        df["end"] = pd.to_datetime(df["end"])
        df["filed"] = pd.to_datetime(df["filed"])
        df = df[df["form"] == "10-K"]

        # Income and cash flow items cover a period: keep full years only
        if "start" in df.columns:
            df["start"] = pd.to_datetime(df["start"])
            df = df[(df["end"] - df["start"]).dt.days.between(350, 380)]

        # Keep only periods ending in the company's fiscal year-end month
        df = df[df["end"].dt.month == fye_month]

        df["tag"] = tag
        df["priority"] = priority
        tables.append(df)

    if not tables:
        return None   # none of the candidate tags were found

    combined = pd.concat(tables)
    # For each date: most preferred tag first, then the most recently filed version
    combined = combined.sort_values(["priority", "filed"], ascending=[True, False])
    combined = combined.drop_duplicates(subset="end", keep="first")

    combined["value_000s"] = combined["val"] / 1000
    return combined.sort_values("end")[["end", "value_000s", "tag", "accn"]]


# Each variable from the data dictionary, with the XBRL tags to try (in priority order)
TAGS = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues"],
    "operating_income": ["OperatingIncomeLoss"],
    "interest_expense": ["InterestExpenseNonoperating", "InterestExpense", "InterestExpenseDebt"],
    "diluted_shares": ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "d_and_a": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization",
                "DepreciationAmortizationAndAccretionNet"],
    "cfo": ["NetCashProvidedByUsedInOperatingActivities"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "acquisitions_paid": ["PaymentsToAcquireBusinessesNetOfCashAcquired"],
    "cash": ["CashAndCashEquivalentsAtCarryingValue"],
    "short_term_debt": ["LongTermDebtCurrent", "DebtCurrent", "ShortTermBorrowings"],
    "long_term_debt": ["LongTermDebtNoncurrent"],
    "finance_lease_current": ["FinanceLeaseLiabilityCurrent"],
    "finance_lease_noncurrent": ["FinanceLeaseLiabilityNoncurrent"],
    "stockholders_equity": ["StockholdersEquity"],
    "accounts_receivable": ["AccountsReceivableNetCurrent",
                            "AccountsReceivableExcludingAccruedInterestAfterAllowanceForCreditLossCurrent",
                            "ReceivablesNetCurrent"],
    "goodwill": ["Goodwill"],
}

# Fiscal year-end month: taken from the most recent annual revenue period (cell 3)
fye_month = annual["end"].iloc[-1].month
print("Fiscal year-end month:", fye_month)

Fiscal year-end month: 12


In [17]:
# FY2025 values from the Excel answer key ($000s; diluted shares in thousands)
answer_key_2025 = {
    "revenue": 724024, "operating_income": 40572, "interest_expense": 14643,
    "diluted_shares": 32058, "d_and_a": 31008, "cfo": 32981, "capex": 24674,
    "acquisitions_paid": 0, "cash": 28008, "short_term_debt": 12849,
    "long_term_debt": 165143, "finance_lease_current": 7025,
    "finance_lease_noncurrent": 17340, "stockholders_equity": 235104,
    "accounts_receivable": 154673, "goodwill": 184829,
}

results = []
for variable, tags in TAGS.items():
    expected = answer_key_2025[variable]
    table = get_annual_values(data, tags, fye_month)

    if table is None:
        results.append([variable, None, expected, None, "TAG NOT FOUND"])
        continue

    row = table[table["end"].dt.year == 2025]
    if row.empty:
        results.append([variable, table["tag"].iloc[0], expected, None, "NO FY2025 VALUE"])
        continue

    api_value = row["value_000s"].iloc[0]
    status = "MATCH" if abs(api_value - expected) < 1 else "MISMATCH"
    results.append([variable, row["tag"].iloc[0], expected, api_value, status])

check = pd.DataFrame(results, columns=["variable", "tag_used", "answer_key", "api_value", "status"])
check

,variable,tag_used,answer_key,api_value,status
0,revenue,RevenueFromContractWithCustomerExcludingAssess...,724024,724024.0,MATCH
1,operating_income,OperatingIncomeLoss,40572,40572.0,MATCH
2,interest_expense,InterestExpenseNonoperating,14643,14643.0,MATCH
3,diluted_shares,WeightedAverageNumberOfDilutedSharesOutstanding,32058,32058.0,MATCH
4,d_and_a,DepreciationDepletionAndAmortization,31008,31008.0,MATCH
5,cfo,NetCashProvidedByUsedInOperatingActivities,32981,32981.0,MATCH
6,capex,PaymentsToAcquirePropertyPlantAndEquipment,24674,24674.0,MATCH
7,acquisitions_paid,PaymentsToAcquireBusinessesNetOfCashAcquired,0,NaN,NO FY2025 VALUE
8,cash,CashAndCashEquivalentsAtCarryingValue,28008,28008.0,MATCH
9,short_term_debt,LongTermDebtCurrent,12849,12849.0,MATCH


In [18]:
def find_tags_with_value(data, target_value, year):
    """Search every reported fact for a given value in a given year's 10-K data."""
    matches = set()
    for taxonomy, tags in data["facts"].items():          # e.g. "us-gaap", "dei"
        for tag, info in tags.items():
            for unit_name, entries in info["units"].items():
                for entry in entries:
                    if (entry.get("form") == "10-K"
                            and entry["end"].startswith(str(year))
                            and entry["val"] == target_value):
                        matches.add((taxonomy, tag, entry["end"]))
    return sorted(matches)

print("Interest expense (14,643,000):")
for match in find_tags_with_value(data, 14643000, 2025):
    print("  ", match)

print("Accounts receivable (154,673,000):")
for match in find_tags_with_value(data, 154673000, 2025):
    print("  ", match)

Interest expense (14,643,000):
   ('us-gaap', 'InterestExpenseNonoperating', '2025-12-31')
Accounts receivable (154,673,000):
   ('us-gaap', 'ReceivablesNetCurrent', '2025-12-31')


In [19]:
   get_annual_values(data, TAGS["interest_expense"], fye_month).tail(5)

,end,value_000s,tag,accn
56,2021-12-31,10882.0,InterestExpense,0001436126-24-000025
0,2022-12-31,10505.0,InterestExpenseNonoperating,0001436126-25-000026
6,2023-12-31,16761.0,InterestExpenseNonoperating,0001628280-26-016765
17,2024-12-31,17067.0,InterestExpenseNonoperating,0001628280-26-016765
26,2025-12-31,14643.0,InterestExpenseNonoperating,0001628280-26-016765
